In [127]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt 

In [128]:
data = pd.read_csv('Datasets/train.csv')

data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [129]:
data = np.array(data)

m, n = data.shape # m = number of data, n = number of features + 1(label)
np.random.shuffle(data)

# Training on 1000 datasets and leaving the rest for testing
data_dev = data[0:1000].T
Y_dev = data_dev[0] # labels
X_dev = data_dev[1:n]
X_dev = X_dev / 255 # scaling the data into a range of 0 to 1. May not be necessary I think.

data_train = data[1000:m].T
Y_train = data_train[0]
X_train = data_train[1:n]
X_train = X_train / 255

In [130]:
data

array([[6, 0, 0, ..., 0, 0, 0],
       [6, 0, 0, ..., 0, 0, 0],
       [6, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(42000, 785))

In [131]:
X_train

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(784, 41000))

In [ ]:
def init_params():
    W1 = np.random.rand(10, 784) - 0.5
    b1 = np.random.rand(10, 1) - 0.5
    W2 = np.random.rand(10, 10) - 0.5 
    b2 = np.random.rand(10, 1) - 0.5

    return W1, b1, W2, b2

def ReLU(Z):
    return np.maximum(0, Z)

def softmax(Z):
    Z = Z - Z.max(1, keepdims = True)
    e = np.exp(Z)
    return e / e.sum(1, keepdims = True)

def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1)) # size 0 to m and 9 + 1
    one_hot_Y[np.arange(Y.size), Y] = 1 # specifying the required column to be 1. 
    one_hot_Y = one_hot_Y.T
    return one_hot_Y

def deriv_ReLU(Z): # slope is 1 if value is greater than 0 so this part is very simple.
    return Z > 0

def back_prop(Z1, A1, Z2, A2, W2, X, Y):
    one_hot_Y = one_hot(Y)
    
    dZ2 = A2 - one_hot_Y
    dW2 = 1 / m * dZ2.dot(A1.T)
    db2 = 1 / m * np.sum(dZ2) 

    dZ1 = W2.T.dot(dZ2) * deriv_ReLU(Z1)
    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1) 

    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2
    return W1, b1, W2, b2

In [133]:
def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, iterations, alpha):
    W1, b1, W2, b2 = init_params()

    for i in range(iterations):
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = back_prop(Z1, A1, Z2, A2, W2, X, Y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
        if i % 50 == 0:
            print("Iteration: ", i)
            print("Accuracy: ", get_accuracy(get_predictions(A2), Y))

    return W1, b1, W2, b2

In [134]:
W1, b1, W2, b2 = gradient_descent(X_train, Y_train, 500, .1)

Iteration:  0
[2 7 8 ... 2 0 1] [0 8 7 ... 0 0 0]
Accuracy:  0.08924390243902439


C:\Users\sudha\AppData\Local\Temp\ipykernel_36640\3596461253.py:13: RuntimeWarning: overflow encountered in exp
  e = np.exp(Z)
C:\Users\sudha\AppData\Local\Temp\ipykernel_36640\3596461253.py:14: RuntimeWarning: invalid value encountered in divide
  return e / e.sum(1, keepdims = True)


Iteration:  50
[0 0 0 ... 0 0 0] [0 8 7 ... 0 0 0]
Accuracy:  0.09841463414634147
Iteration:  100
[0 0 0 ... 0 0 0] [0 8 7 ... 0 0 0]
Accuracy:  0.09841463414634147
Iteration:  150
[0 0 0 ... 0 0 0] [0 8 7 ... 0 0 0]
Accuracy:  0.09841463414634147


KeyboardInterrupt: 